# Fair Gaussian processes with noiseless MAP prediction

This example compares five fixed correlations $\rho\in\{-1,-0.9,0,0.9,1\}$ and a correlation $\rho^*$ fitted on the training set. Each model is evaluated along both an MMD and a Sinkhorn penalty path. Run the cells in order from a checkout with the dependencies in `requirements.txt` installed.

The kernel factors as $k((x,s),(x',s'))=k_x(x,x')B_{ss'}$, where $B_{00}=B_{11}=1$ and $B_{01}=B_{10}=\rho$. At $\rho=1$ the prediction is invariant to changing $s$ at a fixed $x$. At $\rho=0$ the group functions have zero prior cross-covariance and share spatial hyperparameters. Negative $\rho$ gives negative prior cross-covariance.

The optimization transports the latent training values $z$ by minimizing $(z-\mu)^\top\Gamma^{-1}(z-\mu)+\lambda D(z_0,z_1)$. The notebook exports train and test metrics for each fitted model and penalty. Its iteration budgets are small enough for a CPU demonstration.

In [ ]:
from pathlib import Path
import sys
import time

root = next((path for path in (Path.cwd(), *Path.cwd().parents)
             if (path / 'Fair_GPs' / 'Fair_GPs_classe.py').is_file()), None)
if root is None:
    raise RuntimeError("Start Jupyter from the repository or its notebook directory.")
sys.path.insert(0, str(root))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch as t
from geomloss import SamplesLoss
from IPython.display import display

from Fair_GPs.Fair_GPs_classe import GP, predict_map_path
from Fair_GPs.Optim_Method import Optimize_fair_quad, mmd_loss
from Fair_GPs.Kernel_classe import MaternKernel
from Fair_GPs.Data_Processing import normalize_train_test
from Fair_GPs.Data_Processing import SolutionAnalyzer
from Fair_GPs.utils import MSE, fair_regression, ks_from_empirical_cdfs

t.set_default_dtype(t.float64)
t.set_num_threads(2)
device = t.device('cpu')
SEED = 42
FIXED_RHOS = [-1.0, -0.9, 0.0, 0.9, 1.0]
GP_ITERS = 500  # Iterations per hyperparameter fit.
N_STARTS = 3
START_DESIGN = 'sobol'  # Use 'lhs' for a Latin hypercube.
PREDICTION_JITTER = 0.0
out = root / 'results' / 'fair_gp_example'

## 1. Data and train-fitted normalization

The two groups have different spatial input distributions and a shared raw loan function $3x_1+6x_2$, with an added offset of 10 for group 0. The train set has 100 points per group. The independently generated test set has 1,000 points per group.

Fit spatial min/max bounds and target centering and scaling on the training data. The target scale is the largest absolute deviation from its training mean. Both splits use the same transformation and clipping rules. Spatial features lie in `[0, 1]`, and targets lie in `[-1, 1]`. Training targets have mean zero. The sensitive label stays binary.

The clipping report counts values outside these ranges before clipping. Metrics use the centered target scale. Multiply predictions by `normalization['y_scale']` and add `normalization['y_mean']` to express them in the original units.

In [ ]:
def loan(x):
    return 3*x[:, 0] + 6*x[:, 1]

def generate(n0, n1, seed, training=False):
    t.manual_seed(seed)
    mean0 = t.tensor([3, 3], device=device)
    L0 = t.linalg.cholesky(t.tensor([[0.9, 0.3], [0.3, 0.6]], device=device))
    if training:
        t.randn(n0, 2, device=device)  # Preserve the training seed sequence.
    x0 = t.randn(n0, 2, device=device) @ L0.T + mean0
    l0 = loan(x0) + 10
    u = t.rand(n1, device=device)*5 + 0.5
    v = t.randn(n1, device=device)*0.2
    L1 = t.linalg.cholesky(t.tensor([[0.3, 0.1], [0.1, 0.1]], device=device))
    x1 = t.stack((u, 2 + 2/u + v), dim=1) + t.randn(n1, 2, device=device) @ L1.T
    l1 = loan(x1)
    x = t.cat((x0, x1))
    s = t.cat((t.zeros(n0, device=device), t.ones(n1, device=device)))
    xs = t.cat((x, s[:, None]), dim=1)
    return xs, t.cat((l0, l1)), s

n0, n1 = 100, 100  # Training observations per group.
m0, m1 = 1000, 1000  # Test observations per group (2,000 total).
XS_train, Y_train, S = generate(n0, n1, SEED, training=True)
XS_test, Y_test, S_test = generate(m0, m1, SEED + 1)
XS_train_raw, Y_train_raw = XS_train.clone(), Y_train.clone()
XS_test_raw, Y_test_raw = XS_test.clone(), Y_test.clone()
XS_train, Y_train, XS_test, Y_test, normalization = normalize_train_test(
    XS_train_raw, Y_train_raw, XS_test_raw, Y_test_raw, center_targets=True)
display(pd.DataFrame(normalization['clipping_report']))
assert t.isclose(Y_train.mean(), Y_train.new_zeros(()), rtol=0, atol=1e-12)
for split, xs, yy in [('train', XS_train, Y_train), ('test', XS_test, Y_test)]:
    assert ((xs[:, :-1] >= 0) & (xs[:, :-1] <= 1)).all()
    assert ((yy >= -1) & (yy <= 1)).all()
    print(f'{split}: X=[{xs[:, :-1].min():.3f}, {xs[:, :-1].max():.3f}], '
          f'y=[{yy.min():.3f}, {yy.max():.3f}], mean(y)={yy.mean():.6f}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for group in (0, 1):
    mask = S == group
    axes[0].scatter(*XS_train[mask, :2].T.numpy(), label=f'S={group}', alpha=0.6)
    yy = t.sort(Y_train[mask].flatten()).values.numpy()
    axes[1].step(np.r_[yy[0], yy], np.r_[0, np.arange(1, len(yy)+1)/len(yy)], where='post', label=f'S={group}')
axes[0].set(xlabel='Normalized salary', ylabel='Normalized wealth', title='Populations')
axes[0].set_xlim(-0.03, 1.03); axes[0].set_ylim(-0.03, 1.03)
axes[1].set(xlabel='Centered and scaled loan', ylabel='CDF', title='Target distributions')
for ax in axes:
    ax.legend()
fig.tight_layout()
plt.show()

## 2. Estimate the GP hyperparameters

Each fixed-$\rho$ model fits its other parameters by training marginal likelihood. The fitted-$\rho$ model uses the current initialization and additional Sobol starts that vary the trainable kernel and noise parameters jointly. The GP keeps the start with the highest training marginal likelihood. `GP_ITERS` applies to each start.

$\rho$ describes prior dependence at matched spatial inputs. The two target functions in this example differ by a constant offset and have the same slopes. Their centered group means summarize target levels, while $\rho$ controls the covariance of latent deviations around the GP mean.

After fitting, the example sets the observation variance to the sum of training MSE and the weighted squared Wasserstein distance to the group-prediction barycenter. The resulting GP supplies the conditional mean $\mu$ and precision $\Gamma^{-1}$ for the transport problem. `GP.get_inv_cond_gram` uses the likelihood-added conditional covariance with diagonal jitter of `1e-8`. The multistart table records the training likelihood before this noise adjustment.

In [ ]:
models, posterior, hyperparams = {}, {}, []
rho_fit_table = None
for key in [*FIXED_RHOS, 'rho_star']:
    estimated = key == 'rho_star'
    t.manual_seed(SEED)
    gp = GP(XS_train, Y_train, kernel_x='matern52',
            rho=0.0 if estimated else key, device=device)
    gp.Optimise_hyperparameters(
        fix_rho=not estimated,
        iters=GP_ITERS,
        n_starts=N_STARTS if estimated else 1,
        design=START_DESIGN,
        seed=SEED,
        verbose=False,
    )
    if estimated:
        rows = []
        for result in gp.multistart_results:
            initial = {f'initial_{name}': value for name, value in result['initial'].items()}
            fitted = {f'fitted_{name}': value for name, value in result['fitted'].items()}
            rows.append({
                'start_index': result['start_index'],
                'negative_mll': result['loss'],
                'selected': result['selected'],
                **initial, **fitted,
            })
        rho_fit_table = pd.DataFrame(rows)
    varnoise_mll = gp.hyperparameters.varnoise
    pred = gp.get_cond_mean(XS_train).reshape(-1)
    variance = float(fair_regression(pred[S == 0], pred[S == 1])[2] +
                     MSE(pred, Y_train.reshape(-1)))
    gp.set_hyperparameters(varnoise=variance)
    gp.model.train()
    gp.model.eval()
    mu = gp.get_cond_mean(XS_train)[:, None]
    gamma_inv = gp.get_inv_cond_gram(XS_train)
    models[key] = gp
    posterior[key] = (mu, gamma_inv)
    hyperparams.append({'model_key': str(key), 'rho_estimated': estimated,
                        'varnoise_mll': varnoise_mll, **gp.hyperparameters.as_dict()})

rho_star = models['rho_star'].hyperparameters.rho
MODEL_KEYS = sorted(models, key=lambda key: (models[key].hyperparameters.rho, key == 'rho_star'))
RHOS = [models[key].hyperparameters.rho for key in MODEL_KEYS]
RHO_LABELS = [f'ρ*={rho:.6f}' if key == 'rho_star' else f'{rho:g}'
              for key, rho in zip(MODEL_KEYS, RHOS)]
model_metadata = pd.DataFrame({
    'rho_index': range(len(RHOS)),
    'model_key': [str(key) for key in MODEL_KEYS],
    'rho': RHOS,
    'rho_label': RHO_LABELS,
    'rho_estimated': [key == 'rho_star' for key in MODEL_KEYS],
})
hyperparams_table = pd.DataFrame(hyperparams).sort_values(['rho', 'rho_estimated'])
print(f'Estimated rho = {rho_star:.6f}')
if rho_fit_table is not None and not rho_fit_table.empty:
    display(rho_fit_table)
display(hyperparams_table)

## 3. Choose penalty weights and optimize

For each model, the first penalty is zero. The remaining weights are $\lambda_j=\|\Gamma^{-1}\|_F4^{t_j}$, with $t_j$ evenly spaced between the endpoints in `settings`. The positive weights use budgets that increase linearly from `min_iter` to `max_iter`. Table columns show the relative weights $4^{t_j}$, and the CSV records the absolute weights for each model.

Each method starts from $\mu$ at zero penalty. The previous solution initializes the next optimization along that method's path. MMD uses a sum of two Matérn kernels. Sinkhorn uses $p=2$, `blur=0.1`, and debiasing. Larger penalties favor lower distribution discrepancy. Short optimization runs can produce nonmonotone metrics.

In [ ]:
k = 6  # Number of positive penalty values.
if not isinstance(k, int) or k < 2:
    raise ValueError('Choose at least two positive penalty weights.')
settings = {
    'MMD': {'a': -2, 'b': 4, 'min_iter': 50, 'max_iter': 270},
    'Sinkhorn': {'a': -1, 'b': 5, 'min_iter': 30, 'max_iter': 130},
}
lambda_grids, lambda_factors, iteration_budgets = {}, {}, {}
precision_norms = t.stack([t.linalg.norm(posterior[key][1], ord='fro') for key in MODEL_KEYS])
for method, config in settings.items():
    exponents = t.linspace(config['a'], config['b'], k)
    factors = t.cat((t.zeros(1), 4.0 ** exponents))
    lambda_factors[method] = factors
    lambda_grids[method] = precision_norms[:, None] * factors[None, :]
    iteration_budgets[method] = t.linspace(config['min_iter'], config['max_iter'], k).long().tolist()
    display(pd.DataFrame(lambda_grids[method].numpy(),
                         index=pd.Index(RHO_LABELS, name='rho'),
                         columns=[f'lambda_{i}' for i in range(k + 1)]))

### Run the penalty paths

The optimizer records the objective at each step. The table below the metric plots recomputes the divergence at the final latent solution.

In [ ]:
kernel = MaternKernel(lengthscale=0.12, nu=1.5) + MaternKernel(lengthscale=0.4, nu=1.5)
sinkhorn_blur = 0.1
sinkhorn = SamplesLoss(loss='sinkhorn', p=2, blur=sinkhorn_blur, debias=True)
solutions, logs, divergences = {}, {}, {}

for method in settings:
    started = time.perf_counter()
    solutions[method], logs[method], divergences[method] = [], [], []
    for ri, model_key in enumerate(MODEL_KEYS):
        mu, gamma_inv = posterior[model_key]
        path, history = [mu.clone()], [None]
        for li, count in enumerate(iteration_budgets[method], start=1):
            values, _, journal = Optimize_fair_quad(
                X0=path[-1], mu=mu, Gamma=gamma_inv, loss_type=method,
                kernel=kernel, blur=sinkhorn_blur, p=2, lr=8e-3,
                n_iter=count, lambd=lambda_grids[method][ri, li],
                S=S[:, None], autograd=True, verbose=False,
            )
            path.append(values.detach())
            history.append(journal)
        if method == 'MMD':
            distances = [float(mmd_loss(z[S == 0], z[S == 1], kernel)) for z in path]
        else:
            distances = [float(sinkhorn(z[S == 0], z[S == 1])) for z in path]
        solutions[method].append(path)
        logs[method].append(history)
        divergences[method].append(distances)
    print(f'{method} completed in {time.perf_counter() - started:.1f} s')

## 4. Predict and evaluate

The noiseless MAP extension of each latent solution is $K(X_{\rm eval},X_{\rm train})K(X_{\rm train},X_{\rm train})^{-1}z$. The implementation solves the kernel system for all penalty values together. It evaluates the kernel at both sensitive labels for each individual, using the observed label for the primary prediction.

- **MSE** measures squared prediction error on the centered target scale.
- **KS** measures the largest absolute difference between the prediction CDFs of the two observed groups.
- **disp** averages $[f(x,0)-f(x,1)]^2$ over paired predictions for the same individuals.

At $\rho=1$, disp is zero up to numerical precision. KS can remain positive because the two groups have different spatial distributions.

In [ ]:
predictions, counterfactuals, analyzers, tables = {}, {}, {}, []
for method in settings:
    predictions[method], counterfactuals[method] = {}, {}
    for split, xs, targets, groups in [
        ('train', XS_train, Y_train, S),
        ('test', XS_test, Y_test, S_test),
    ]:
        observed, pred0, pred1 = [], [], []
        for key, path in zip(MODEL_KEYS, solutions[method]):
            p, p0, p1 = predict_map_path(models[key], xs, path, jitter=PREDICTION_JITTER)
            observed.append(p)
            pred0.append(p0)
            pred1.append(p1)
        predictions[method][split] = observed
        counterfactuals[method][split] = pred0, pred1
        analyzer = SolutionAnalyzer(observed, lambda_grids[method], groups, targets,
                                    rhos=RHOS, counterfactuals=(pred0, pred1))
        analyzers[method, split] = analyzer
        table = analyzer.compute_metrics().merge(model_metadata.drop(columns='rho'), on='rho_index')
        table['method'], table['split'] = method, split
        table['lambda_relative'] = lambda_factors[method].numpy()[table['lambda_index']]
        tables.append(table)

metrics = pd.concat(tables, ignore_index=True)
assert len(metrics) == len(settings) * 2 * len(RHOS) * (k + 1)
assert np.isfinite(metrics[['MSE', 'KS', 'disp']].to_numpy()).all()
disp_rho1 = metrics.loc[metrics.model_key.eq('1.0'), 'disp']
if not disp_rho1.empty:
    assert disp_rho1.max() < 1e-16
display(metrics.head())

## 5. Compare methods and splits

The four tables show rows ordered by fitted or fixed rho, with the fitted row marked ρ*. Each cell displays MSE in blue, KS in red, and disp in ochre. Symbols also identify the metrics. The penalty columns compare relative weights within each method.

Use a validation set when selecting a model or penalty. The test results illustrate performance for this simulation and seed.

In [ ]:
table_dir = out / 'tables'
table_dir.mkdir(parents=True, exist_ok=True)
for (method, split), analyzer in analyzers.items():
    fig, ax = analyzer.plot_metrics_table(
        title=f'{method} — {split} — centered target scale',
        lambda_factors=lambda_factors[method].numpy(),
        digits=4,
    )
    for ri, label in enumerate(RHO_LABELS):
        row_text = next(text for text in ax.texts if text.get_position() == (0.5, ri + 1.5))
        row_text.set_text(label)
    fig.savefig(table_dir / f'{method}_{split}.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
interpolation_rows = []
for method in settings:
    for ri, key in enumerate(MODEL_KEYS):
        targets = t.stack(solutions[method][ri]).reshape(k + 1, -1)
        errors = (predictions[method]['train'][ri] - targets).abs().amax(dim=1)
        interpolation_rows.extend(
            {'method': method, 'model_key': str(key), 'lambda_index': li,
             'max_interpolation_error': float(error)}
            for li, error in enumerate(errors)
        )
interpolation_checks = pd.DataFrame(interpolation_rows)
display(interpolation_checks.groupby('method')['max_interpolation_error'].max().to_frame())

## 6. Numerical diagnostics and saved results

The curves show the objective along the last positive penalty for each model. The interpolation check above compares training predictions with the transported values. Numerical error depends on the condition of the training kernel. `PREDICTION_JITTER` adds a diagonal regularizer to the interpolation solve when needed.

The notebook saves the metric tables, normalization and fit summaries, diagnostics, and the tensor paths for further analysis under `results/fair_gp_example/`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, method in zip(axes, settings):
    for ri, label in enumerate(RHO_LABELS):
        ax.plot(logs[method][ri][-1]['loss'], label=label)
    ax.set(xlabel='Iteration', ylabel='Objective', title=f'{method}, final penalty', yscale='symlog')
    ax.legend()
fig.tight_layout()
plt.show()

divergence_rows = [
    {'method': method, 'model_key': str(key), 'rho': RHOS[ri],
     'lambda_index': li, 'final_divergence': value,
     'iterations': len(logs[method][ri][li]['loss']) if li else 0}
    for method in settings for ri, key in enumerate(MODEL_KEYS)
    for li, value in enumerate(divergences[method][ri])
]
divergence_table = pd.DataFrame(divergence_rows)
display(divergence_table.loc[divergence_table.lambda_index.eq(k)])

metrics.to_csv(out / 'metrics.csv', index=False)
if rho_fit_table is not None and not rho_fit_table.empty:
    rho_fit_table.to_csv(out / 'rho_fit_starts.csv', index=False)
pd.DataFrame(normalization['clipping_report']).to_csv(out / 'normalization_report.csv', index=False)
model_metadata.to_csv(out / 'rho_models.csv', index=False)
hyperparams_table.to_csv(out / 'hyperparameters.csv', index=False)
interpolation_checks.to_csv(out / 'interpolation_checks.csv', index=False)
divergence_table.to_csv(out / 'divergences.csv', index=False)
t.save({
    'rhos': RHOS, 'lambdas': lambda_grids, 'solutions': solutions,
    'predictions': predictions, 'counterfactuals': counterfactuals,
    'logs': logs, 'normalization': normalization, 'model_keys': MODEL_KEYS,
    'rho_labels': RHO_LABELS, 'rho_star': rho_star, 'n_train': len(S), 'n_test': len(S_test),
    'prediction_jitter': PREDICTION_JITTER,
}, out / 'grid.pt')
print(f'Saved results in {out.relative_to(root)}')

## Further experiments

Try a longer iteration budget or a different start design, then inspect the fitted hyperparameters and the objective curves. To assess robustness, repeat the full fit over several train/test seeds. To isolate the effect of rho, compare models with common spatial hyperparameters and observation variance.

## Independent KS verification

The metric tables use `Fair_GPs.utils.KS`, which calls SciPy's two-sample statistic. The audit uses `Fair_GPs.utils.ks_from_empirical_cdfs` to compute both empirical CDFs on their pooled support and checks every reported KS value. The two calculations use separate implementations in the shared utility module. Equal group sizes give KS increments of 1/100 on train and 1/1,000 on test. The audit also compares the KS of the transported training values with the KS after the noiseless MAP extension.

In [ ]:
audit_rows = []
for method in settings:
    for split, sensitive in [('train', S), ('test', S_test)]:
        for ri, (model_key, rho) in enumerate(zip(MODEL_KEYS, RHOS)):
            path = predictions[method][split][ri]
            for li, prediction in enumerate(path):
                manual_ks = ks_from_empirical_cdfs(prediction, sensitive)
                row = metrics[(metrics.method == method) & (metrics.split == split)
                              & (metrics.rho_index == ri) & (metrics.lambda_index == li)].iloc[0]
                assert np.isclose(manual_ks, row.KS, rtol=0, atol=2e-15)
                previous_ks = ks_from_empirical_cdfs(path[li-1], sensitive) if li else np.nan
                audit_rows.append(dict(method=method, split=split, rho=rho, lambda_index=li,
                    KS=float(row.KS), KS_independent=manual_ks,
                    model_key=str(model_key), rho_estimated=(model_key == 'rho_star'),
                    prediction_max_change=float((prediction-path[li-1]).abs().max()) if li else np.nan,
                    same_KS_as_previous=bool(li and np.isclose(manual_ks, previous_ks, rtol=0, atol=2e-15)),
                    transported_KS=ks_from_empirical_cdfs(solutions[method][ri][li], S) if split == 'train' else np.nan))
ks_audit = pd.DataFrame(audit_rows)
display(ks_audit.head())
ks_audit.to_csv(out / 'ks_audit.csv', index=False)
print(f"Verified {len(ks_audit)} KS values against explicit empirical CDFs.")